In [1]:
import os, shlex, subprocess, json
from datetime import datetime
from pathlib import Path
from typing import Optional, Tuple, Dict
from dotenv import load_dotenv
load_dotenv(Path("configs") / "local.env")

#from data.minbpe import BasicTokenizer as Tokenizer
from src.minbpe import RegexTokenizer as Tokenizer
#from src.gpt import GPTLanguageModel
from src.transformer.model_relative_positional_encoding import GPTLanguageModel

import pandas as pd
import matplotlib.pyplot as plt
import torch
torch.manual_seed(3647)
torch.set_float32_matmul_precision('high')
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader

In [2]:
def now():
    return datetime.now().astimezone().strftime('%FT%T%:z')

def check_ckpt_files(ckpt_dir, match="checkpoint_*.pt"):
    result = {}

    def handle(pt_file):
        tags = str(pt_file).replace("checkpoint_", "").replace(".pt", "").split("-", 1)
        epoch, step = tags[0], int(tags[1])
        if epoch not in result:
            result[epoch] = []

        result[epoch].append((step, pt_file))

    for f in ckpt_dir.glob(match):
        handle(f)

    for _, v in result.items():
        values = sorted(
            v,
            #key=lambda x: x.stat().st_ctime,
            key=lambda x: x[0],
            reverse=True,
        )

        for e in values[1:]:
           print(f"Remove ckpt file: {e[1]}")
           os.remove(e[1])

def send_notification(title, message):
    cmd = os.getenv("send_notification")
    if cmd is None:
        return

    command = shlex.split(cmd)
    command.append(title)
    command.append(message)

    _ = subprocess.Popen(command)


In [3]:
def print_model_structure(model: nn.Module, indent: str = '') -> None:
    """
    Custom function to print model structure in a hierarchical format
    """
    for name, child in model.named_children():
        params = sum(p.numel() for p in child.parameters())
        print(f"{indent}├─ {name}: {child.__class__.__name__} ({params:,} parameters)")
        print_model_structure(child, indent + '│  ')

def get_model_stats(model: torch.nn.Module) -> pd.DataFrame:
    """
    Create a DataFrame with detailed layer statistics
    """
    stats = []
    for name, module in model.named_modules():
        if len(list(module.children())) == 0:  # Only leaf modules
            params = sum(p.numel() for p in module.parameters())
            stats.append({
                'Layer Name': name,
                'Type': module.__class__.__name__,
                'Parameters': params,
                'Trainable': sum(p.numel() for p in module.parameters() if p.requires_grad),
            })

    return pd.DataFrame(stats)

def reset_dropout(model, p):
    for module in model.modules():
        if isinstance(module, nn.Dropout):
            module.p = p

class TextDataset(Dataset):
    def __init__(self, data: torch.Tensor, block_size: int) -> None:
        self.data = data
        self.block_size = block_size

    def __len__(self) -> int:
        return len(self.data) - self.block_size

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, torch.Tensor]:
        x = self.data[index:index + self.block_size]
        y = self.data[index + 1:index + self.block_size + 1]
        return x, y

def get_dataloaders(
        train_data: torch.Tensor,
        val_data: torch.Tensor,
        block_size: int,
        batch_size: int,
        device: torch.device,
) -> Tuple[DataLoader, DataLoader]:
    train_dataset = TextDataset(train_data.to(device), block_size)
    val_dataset = TextDataset(val_data.to(device), block_size)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
    )

    return train_loader, val_loader

@torch.no_grad()
def estimate_loss(model: torch.nn.Module, eval_dataset: Dict[str, DataLoader]) -> Dict[str, float]:
    output = {}
    model.eval() # 评估模式

    for split, loader in eval_dataset.items():
        losses = torch.zeros(eval_batches)
        for i, (x, y) in enumerate(loader):
            with torch.no_grad():
                _, loss = model(x, y)
            losses[i] = loss.item()

        output[split] = float(losses.mean().item())

    model.train() # 切换到为推理
    return output

In [4]:
#### 1. init
run_name = "ch09_train"
device = 'cuda' if torch.cuda.is_available() else 'cpu'

tokenizer_dir = Path("data") / "tokenizer"
checkpoint_dir = Path("data") / "ch09"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

In [5]:
#### 2. setup
tokenizer = Tokenizer()
tokenizer.load(model_file=str(tokenizer_dir / "tokenizer.model"))

train_data = torch.load(tokenizer_dir / 'train.tokens.pt')
val_data = torch.load(tokenizer_dir / 'validation.tokens.pt')

print(f"--> train_data: {train_data.size()[0]:_}, val_data: {val_data.size()[0]:_}")

--> train_data: 12_092_088, val_data: 616_680


In [6]:
#### 3. parameters
##### 3.1 model parameters
parameters = {
    'vocab_size': len(tokenizer.vocab),
    'n_embd': 512,
    'block_size': 256,
    'n_head': 8,
    'n_layer': 4,
    #'dropout': 0.2,
    # when epoch > 10
    'dropout': 0.1,
}

##### 3.2 training parameters
total_epoches = 15
eval_interval = 1_000
eval_batches = 1_000 # 5_000

batch_size = 96 # 64, 32
max_learning_rate = 3e-4
#step=984000
#min_learning_rate = 3e-5
min_learning_rate = 1e-5

In [7]:
#### 4. datasets
train_loader, val_loader = get_dataloaders(
    train_data=train_data,
    val_data=val_data,
    block_size=parameters['block_size'],
    batch_size=batch_size,
    device=device,
)

eval_batches = min(eval_batches, len(val_loader))
eval_dataset = {}

eval_dataset['train'], eval_dataset['val'] = get_dataloaders(
    train_data=train_data[:eval_batches*batch_size],
    val_data=val_data[:eval_batches*batch_size],
    block_size=parameters['block_size'],
    batch_size=batch_size,
    device=device,
)

print(f"{now()} train_batches={len(train_loader):_}, validation_batches={len(val_loader):_}, eval_batches={eval_batches:_}")

2025-09-22T16:19:38%:z train_batches=125_957, validation_batches=6_422, eval_batches=1_000


In [8]:
#### 5. Scheduler calculation
gradient_accumulation_steps = 8
last_epoch, last_step = 1, 0
epoch_steps = len(train_loader)
total_steps =  epoch_steps * total_epoches
optimizer_steps_total = total_steps // gradient_accumulation_steps
warmup_iters = int(0.1 * optimizer_steps_total)

print(
    f"{now()} epoch_steps={epoch_steps:_}, total_steps={total_steps:_}, "
    f"optimizer_steps_total={optimizer_steps_total:_}, "
    f"warmup_iters={warmup_iters:_}"
)

2025-09-22T16:19:38%:z epoch_steps=125_957, total_steps=1_889_355, optimizer_steps_total=236_169, warmup_iters=23_616


In [9]:
#### 6. llm
model = GPTLanguageModel(
    vocab_size=parameters['vocab_size'],
    n_embd=parameters['n_embd'],
    block_size=parameters['block_size'],
    n_head=parameters['n_head'],
    n_layer=parameters['n_layer'],
    dropout=parameters['dropout'],
    device=device,
).to(device)

model = torch.compile(model)
optimizer = torch.optim.AdamW(model.parameters(), lr=max_learning_rate)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer=optimizer,
    T_max=optimizer_steps_total - warmup_iters,
    eta_min=min_learning_rate,
)
# scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=6, T_mult=2, eta_min=1e-6)

parameters_m = sum(p.numel() for p in model.parameters())/1e6
print(f'Model parameters: {parameters_m:.3f}M ')

# warmup_scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.01, total_iters=10)
# warmup_scheduler.step()

Model parameters: 13.660M 


In [10]:
#### 7. last checkpoints
ckpt_files = sorted(
    checkpoint_dir.glob("checkpoint_*.pt"),
    #key=lambda x: x.stat().st_ctime,
    key=lambda x: int(x.name.replace("checkpoint_", "").replace(".pt", "").split("-")[-1]),
    reverse=True,
)

if len(ckpt_files) > 0:
    checkpoint_path = ckpt_files[0]
    last_ckpt = torch.load(checkpoint_path, map_location=device) # weights_only=True
    last_epoch = last_ckpt['meta']['epoch']
    last_step = last_ckpt['meta']['step']
    if last_step % len(train_loader) == 0:
        last_epoch += 1

    print(f"load: last_checkpoint={checkpoint_path}, last_step={last_step:07_}")    
    model.load_state_dict(last_ckpt['model_state_dict'])

    optimizer.load_state_dict(last_ckpt['optimizer_state_dict'])

    scheduler.load_state_dict(last_ckpt['scheduler_state_dict'])
    scheduler.T_max = optimizer_steps_total - warmup_iters
    scheduler.eta_min = min_learning_rate

load: last_checkpoint=data/ch09/checkpoint_012-1511484.pt, last_step=1_511_484


In [11]:
# when epoch > 10
reset_dropout(model, 0.1)

In [12]:
#### 8. estimate losses
def estimate_and_save(epoch, step):
    t0 = datetime.now()
    losses = estimate_loss(model, eval_dataset)
    #current_lr = scheduler.get_last_lr()[0]
    current_lr = optimizer.param_groups[0]['lr']

    # Save checkpoint
    checkpoint_prefix = str(checkpoint_dir / f"checkpoint_{epoch:03d}-{step:06d}")

    meta = {
        'created_at': now(),
        'parameters': parameters,
        'epoch': epoch,
        'step': step,
        'learning_rate': current_lr,
        'train_loss': losses['train'],
        'val_loss': losses['val'],

        'run_name': run_name,
        'batch_size': batch_size,
        'n_batch': len(train_loader),
    }

    checkpoint = {
        'meta': meta,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
    }

    with open(checkpoint_prefix + ".json", 'w', encoding='utf-8') as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)
        f.write("\n")

    torch.save(checkpoint, checkpoint_prefix+".pt")
    check_ckpt_files(checkpoint_dir)

    elapsed = datetime.now() - t0

    print(
        f"\n{now()} epoch={epoch}/{total_epoches}, step={step:07_}/{total_steps:07_}, "
        f"\n    train_loss={losses['train']:.3f}, val_loss={losses['val']:.3f}, learning_rate={current_lr:.6f}, "
        f"checkpoint={checkpoint_prefix}.pt, elapsed={str(elapsed)}"
    )

    return losses

In [ ]:
#### 9. training
batch_processed = (last_epoch - 1) * len(train_loader)
optimizer_processed = last_step // gradient_accumulation_steps

print(
    f"{now()} Running: last_epoch={last_epoch}/{total_epoches}, "
    f"last_step={last_step:_}/{total_steps:_}, optimizer_processed={optimizer_processed:_}"
)

train_losses, val_losses = [], []

for epoch in range(last_epoch, total_epoches+1):
    msg = f"{now()} starting epoch: epoch={epoch}/{total_epoches}, last_step={last_step:_}/{total_steps:_}"
    print(msg)
    send_notification(run_name, msg)

    for idx, (x_batch, y_batch) in enumerate(train_loader):
        batch_processed += 1
        if batch_processed <= last_step:
            continue

        # Training step
        optimizer.zero_grad(set_to_none=True)
        logits, loss = model(x_batch, y_batch)
        loss /= gradient_accumulation_steps
        loss.backward()
        #batch_loss = loss.item()

        if batch_processed % gradient_accumulation_steps != 0:
            continue

        optimizer_processed += 1
        if optimizer_processed <= warmup_iters:
            learning_rate = max_learning_rate * (optimizer_processed / warmup_iters)
            for param_group in optimizer.param_groups:
                param_group['lr'] = learning_rate

        optimizer.step()

        if optimizer_processed >= warmup_iters:
            scheduler.step()

        if optimizer_processed % eval_interval != 0:
            continue

        # current_lr = scheduler.get_last_lr()[0]
        current_lr = optimizer.param_groups[0]['lr']
        losses = estimate_and_save(epoch, step=batch_processed)
        train_losses.append(losses['train'])
        val_losses.append(losses['val'])

        epoch_progress = f"{(idx+1)/epoch_steps:.3f}"
        total_progress = f"{batch_processed/total_steps:.3f}"

        msg = f"{now()} evaluation: epoch={epoch}/{total_epoches}, epoch_progress={epoch_progress}, " + \
            f"total_progress={total_progress}, learning_rate={current_lr:.6f}, " + \
            f"train_loss={losses['train']:.3f}, validation_loss={losses['val']:.3f}"

        send_notification(run_name, msg)

    epoch_pt = checkpoint_dir / f"checkpoint_{epoch:03d}-{batch_processed:06d}.pt"
    if not epoch_pt.exists():
        current_lr = optimizer.param_groups[0]['lr']
        losses = estimate_and_save(epoch, step=batch_processed)
        train_losses.append(losses['train'])
        val_losses.append(losses['val'])

send_notification(run_name, "Finished")

2025-09-22T16:19:40%:z Running: last_epoch=13/15, last_step=1_511_484/1_889_355, optimizer_processed=188_935
2025-09-22T16:19:40%:z starting epoch: epoch=13/15, last_step=1_511_484/1_889_355
{"status":1,"request":"b3d65df9-c06b-4cc4-bab6-fca3b740459b"}
2025-09-22T16:23:40%:z epoch=13/15, step=1_512_000/1_889_355, 
    train_loss=0.988, val_loss=2.683, learning_rate=0.000013, checkpoint=data/ch09/checkpoint_013-1512000.pt, elapsed=0:02:06.271643


curl: (28) Operation timed out after 10001 milliseconds with 0 bytes received


{"status":1,"request":"fad2ffa2-cd70-441d-9990-706dc7e236e5"}

In [ ]:
#### 10. data viz
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label="Training Loss") # marker='o'
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Evaluation Step")
plt.ylabel("Loss")
plt.title("Training and Validation Loss Over Time")
plt.legend()
plt.grid()
plt.show()